# 🏗️ CNN Architecture — Notes + Interview
---
> **Simple English** | **Interview Ready**

## 📌 What is CNN Architecture? (Simple English)
- CNN = stack of layers in a specific order to process images
- Classic pattern: **[Conv → Activation → Pool]** × N → **Flatten** → **Dense** → **Output**
- Each block extracts features at different scales
- Depth increases (more channels), spatial size decreases as we go deeper
- Final Dense layers do the actual classification

## 🔑 Standard CNN Pattern
```
Input Image (H×W×C)
    ↓
[CONV(filters) + ReLU + MaxPool] × repeat
    ↓
Flatten (or GlobalAvgPool)
    ↓
Dense(128) + ReLU + Dropout
    ↓
Dense(num_classes) + Softmax
```

## 🧱 Why This Pattern Works
| Block | Purpose |
|---|---|
| Conv + ReLU | Extract features, add non-linearity |
| MaxPool | Reduce size, keep important features |
| Dropout | Prevent overfitting |
| Dense | Combine features for final decision |
| Softmax | Convert to probabilities |

In [ ]:
import tensorflow as tf
import numpy as np

# ── Full CNN for MNIST (28×28 grayscale, 10 classes) ──
def build_cnn():
    model = tf.keras.Sequential([
        # Block 1: Extract low-level features
        tf.keras.layers.Conv2D(32,(3,3),activation='relu',padding='same',input_shape=(28,28,1)),
        tf.keras.layers.MaxPooling2D(2,2),          # 28→14

        # Block 2: Extract higher-level features
        tf.keras.layers.Conv2D(64,(3,3),activation='relu',padding='same'),
        tf.keras.layers.MaxPooling2D(2,2),          # 14→7

        # Block 3: More complex features
        tf.keras.layers.Conv2D(128,(3,3),activation='relu',padding='same'),

        # Classifier head
        tf.keras.layers.GlobalAveragePooling2D(),   # 7×7×128 → 128
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.Dense(10, activation='softmax')
    ])
    return model

model = build_cnn()
model.summary()

In [ ]:
# Train on MNIST to see it work
(x_train, y_train),(x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = x_train.reshape(-1,28,28,1).astype('float32')/255.0
x_test  = x_test.reshape(-1,28,28,1).astype('float32')/255.0

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history = model.fit(x_train, y_train, epochs=5, batch_size=128,
                    validation_split=0.1, verbose=1)
loss, acc = model.evaluate(x_test, y_test, verbose=0)
print(f"\nTest Accuracy: {acc:.4f}")

In [ ]:
import matplotlib.pyplot as plt

# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12,4))
ax1.plot(history.history['accuracy'], label='Train Acc')
ax1.plot(history.history['val_accuracy'], label='Val Acc')
ax1.set_title('Accuracy'); ax1.legend(); ax1.grid(True)
ax2.plot(history.history['loss'], label='Train Loss')
ax2.plot(history.history['val_loss'], label='Val Loss')
ax2.set_title('Loss'); ax2.legend(); ax2.grid(True)
plt.tight_layout(); plt.show()

## 🗣️ Interview Q&A

**Q: What is the typical CNN architecture pattern?**
> [Conv+ReLU → MaxPool] repeated → Flatten/GAP → Dense → Softmax. Conv layers extract features, pooling reduces size, Dense layers classify.

**Q: Why does spatial size decrease but depth increase?**
> Pooling reduces H and W. Each conv block adds more filters (32→64→128), increasing depth (channels). This trades spatial resolution for semantic richness.

**Q: What is the role of Dropout in CNN?**
> Randomly sets neurons to 0 during training, forcing the network not to rely on specific neurons. Prevents overfitting. Usually added after Dense layers.

**Q: When to use CNN vs other models?**
> CNN → images, spatial data. RNN/LSTM → sequences, text, time series. Transformer → large-scale text/image (modern). Dense ANN → tabular data.